# Solve the Lanchester population system with native HHL

Regenerate the source eight-step finite-difference model with six years between steps. Normalize and pad the forcing vector, build the complete 32-dimensional Hermitian embedding, and use six phase bits and all 64 reciprocal rotations. A native Python quantum function specializes the full unitary, QPE and rotation table.

The local simulator executes the HHL block once. After checking its clean workspace, pass its entire logical state, including all phase and indicator branches, into a second public FlagQuantum Circuit. Append the normalized direct-solve state and execute the two Hadamards and five controlled swaps. The SWAP stage produces 30,000 actual local draws. This staged simulation preserves the unmeasured HHL state and uses separate physical resource counts for the two stages. The indicator-conditioned SWAP estimate and the indicator-one/phase-zero statevector fidelity use distinct conditioning.

The plots compare the native signed amplitudes and population trajectories with a fresh direct solve. Run with FlagQuantum, Matplotlib and Jupyter installed.


## Conditional HHL output and sampled overlap answer different questions

The visible native builder prepares the padded forcing vector, composes
six-bit phase estimation and reciprocal rotations, then uncomputes within/apply.
FlagQuantum constructs controlled evolution, inverse composition and named
registers. Host code performs Hermitian embedding, shift, rescaling and solution
interpretation for the original finite Lanchester linear system.

Inspect success probability, absolute-state error and the separate 30000-draw
SWAP result together. The coherent SWAP stage currently uses explicit Circuit
composition of the native logical state; it is not a single high-level whole
program. A normalized postselected vector is not a deterministic classical
solution readout, and finite phase resolution limits reciprocal accuracy.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QBit
from flagquantum.synth import QNum
from flagquantum.synth import allocate
from flagquantum.synth import apply
from flagquantum.synth import inplace_prepare_amplitudes
from flagquantum.synth import qfunc
from flagquantum.synth import qpe
from flagquantum.synth import select_rotation
from flagquantum.synth import unitary
from flagquantum.synth import within
from typing import Annotated
import torch


def build_preview(
    matrix: torch.Tensor, angles: torch.Tensor, padded: torch.Tensor
) -> synth.Program:
    """Bind fresh 32-state matrix and all 64 controlled rotations."""
    evolution = tuple(tuple(map(complex, row)) for row in matrix)
    amplitudes = tuple(map(float, padded))
    rotations = tuple(map(float, angles))

    @qfunc
    def hhl(
        amplitudes: list[float],
        indicator: Output[QBit],
        time_index: Output[Annotated[QNum, 4]],
        rabbits: Output[QBit],
        rescaled_eig: Output[Annotated[QNum, (6, False, 6)]],
    ) -> None:
        allocate(time_index)
        allocate(rabbits)
        allocate(indicator)
        allocate(rescaled_eig)
        inplace_prepare_amplitudes(amplitudes, 0, (rabbits, time_index))
        with within():
            qpe(lambda: unitary(evolution, (rabbits, time_index)), rescaled_eig)
        with apply():
            select_rotation("Y", rotations, rescaled_eig, indicator)

    return synth.synthesize(
        hhl,
        classical_inputs={"amplitudes": amplitudes},
        max_workspace=4,
        max_qubits=18,
        max_operations=25_000,
        memory_limit_bytes=16 * 1024**2,
    )


from importlib import import_module
from pathlib import Path

import torch

from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.main import (
    OUTPUT,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.model import (
    DT,
    PHASE_WORDS,
    N,
    build_model,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.plots import (
    save_matrix_views,
    save_populations,
    save_vectors,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.quantum import (
    SHOTS,
    sampled_swap,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.workflow import (
    show_result,
    solve,
)


def show(path: Path) -> None:
    display = import_module("IPython.display")
    display.display(display.Image(filename=str(path)))


## Regenerate the population model

In [ ]:
model = build_model()
print("Source shape:", model.operator.shape, "sample spacing:", DT)


In [ ]:
show(
    save_matrix_views(
        (
            ("Original operator", model.operator),
            ("Forcing vector", model.forcing[:, None]),
        ),
        OUTPUT / "original-system.png",
    )
)


In [ ]:
direct_population = torch.linalg.solve(model.operator, model.forcing)


In [ ]:
show(
    save_vectors(
        (("Direct classical", direct_population),), OUTPUT / "classical-vector.png"
    )
)


In [ ]:
show(save_populations(direct_population, OUTPUT / "classical-populations.png"))


## Embed the system in a Hermitian matrix

In [ ]:
forcing_norm = model.forcing.norm()
print("Forcing norm:", float(forcing_norm))


In [ ]:
show(
    save_matrix_views(
        (
            ("Hermitian embedding", model.hermitian),
            ("Normalized padded vector", model.padded[:, None]),
        ),
        OUTPUT / "hermitian-system.png",
    )
)


In [ ]:
torch.testing.assert_close(model.rescaled, model.rescaled.T)
assert float(model.phase_eigenvalues.min()) >= 0
assert float(model.phase_eigenvalues.max()) < 1
show(
    save_matrix_views(
        (("Shifted and rescaled operator", model.rescaled),),
        OUTPUT / "shifted-system.png",
    )
)


## Native HHL and coherent SWAP test

In [ ]:
assert model.angles.numel() == PHASE_WORDS
print("Reciprocal rotation entries:", model.angles.numel())


In [ ]:
direct_embedding = torch.linalg.solve(model.hermitian, model.padded)
torch.testing.assert_close(direct_embedding[: 2 * N] * forcing_norm, direct_population)


In [ ]:
assert model.evolution.shape == (4 * N, 4 * N)
print("Fresh unitary shape:", model.evolution.shape)


In [ ]:
print("Source swap-test width bound:", 28, "source shots:", SHOTS)


In [ ]:
solution = solve(model)
print(
    "Native width and operation count:",
    solution.resources.n_qubits,
    solution.resources.n_operations,
)


In [ ]:
swap = sampled_swap(solution.logical, solution.reference)


In [ ]:
print("Sampled overlap after indicator postselection:", swap.root_overlap)


In [ ]:
print("Native coherent SWAP stage: five controlled swaps and two Hadamards")


## Read out the HHL solution

In [ ]:
print("Named program outputs:", tuple(solution.program.registers))


In [ ]:
assert solution.logical.shape == (2, 32, 64)


In [ ]:
print(
    "Basis words represented in the postselected solution:",
    solution.statevector.numel(),
)


In [ ]:
show(
    save_vectors(
        (
            ("Direct classical", solution.reference),
            ("Native HHL", solution.statevector),
        ),
        OUTPUT / "hhl-vector.png",
    )
)


In [ ]:
show(
    save_populations(
        direct_population,
        OUTPUT / "hhl-populations.png",
        solution.statevector * forcing_norm,
    )
)


In [ ]:
show_result(solution, swap)


preview = build_preview(model.evolution, model.angles, model.padded)
print("Editable HHL resources:", preview.report["resources"])
